In [1]:
from pop import Pilot
import numpy as np
import time

Car = Pilot.AutoCar()

In [6]:
dataset={ 'gyro' : [], 'steer' : [] }
Car.setSpeed(70)
for n in np.arange(-1, 1.1, 0.2):
	n = round(n,1)

	Car.steering = n
	Car.forward(70)

	time.sleep(0.5)

	m = Car.getGyro("z")

	time.sleep(0.5)

	Car.backward()

	time.sleep(1)

	Car.stop()

	dataset["gyro"].append(m)
	dataset["steer"].append(n)

	print({ "gyro" : m , "steer" : n })

{'gyro': 2092, 'steer': -1.0}
{'gyro': 2430, 'steer': -0.8}
{'gyro': 2421, 'steer': -0.6}
{'gyro': 1470, 'steer': -0.4}
{'gyro': 840, 'steer': -0.2}
{'gyro': 509, 'steer': -0.0}
{'gyro': -356, 'steer': 0.2}
{'gyro': -664, 'steer': 0.4}
{'gyro': -1096, 'steer': 0.6}
{'gyro': -1915, 'steer': 0.8}
{'gyro': -2159, 'steer': 1.0}


In [8]:
from pop import AI

LR=AI.Linear_Regression()
LR.X_data=dataset["gyro"]
LR.Y_data=dataset["steer"]

LR.train(times=5000, print_every=100)

100 step loss: 795161.9375
200 step loss: 46849.390625
300 step loss: 924.653564453125
400 step loss: 7.3285746574401855
500 step loss: 1.7893481254577637
600 step loss: 1.7558842897415161
700 step loss: 1.728153944015503
800 step loss: 1.698067545890808
900 step loss: 1.6657607555389404
1000 step loss: 1.6313605308532715
1100 step loss: 1.5949867963790894
1200 step loss: 1.556758165359497
1300 step loss: 1.516794204711914
1400 step loss: 1.4752171039581299
1500 step loss: 1.4321527481079102
1600 step loss: 1.3877308368682861
1700 step loss: 1.3420884609222412
1800 step loss: 1.2953664064407349
1900 step loss: 1.2477141618728638
2000 step loss: 1.1992863416671753
2100 step loss: 1.1502445936203003
2200 step loss: 1.1007558107376099
2300 step loss: 1.0509921312332153
2400 step loss: 1.0011308193206787
2500 step loss: 0.9513534903526306
2600 step loss: 0.9018449187278748
2700 step loss: 0.8527916073799133
2800 step loss: 0.8043773174285889
2900 step loss: 0.756786584854126
3000 step loss

In [11]:
LR.train(times=5000, print_every=100)

5100 step loss: 0.1535501778125763
5200 step loss: 0.14722858369350433
5300 step loss: 0.14201074838638306
5400 step loss: 0.13775819540023804
5500 step loss: 0.1343381255865097
5600 step loss: 0.13162581622600555
5700 step loss: 0.12950624525547028
5800 step loss: 0.12787538766860962
5900 step loss: 0.12664085626602173
6000 step loss: 0.125722274184227
6100 step loss: 0.12505100667476654
6200 step loss: 0.12456972151994705
6300 step loss: 0.1242315024137497
6400 step loss: 0.12399878352880478
6500 step loss: 0.12384218722581863
6600 step loss: 0.12373924255371094
6700 step loss: 0.1236732080578804
6800 step loss: 0.12363195419311523
6900 step loss: 0.12360687553882599
7000 step loss: 0.12359204888343811
7100 step loss: 0.12358356267213821
7200 step loss: 0.12357886135578156
7300 step loss: 0.12357635051012039
7400 step loss: 0.12357502430677414
7500 step loss: 0.1235743835568428
7600 step loss: 0.12357406318187714
7700 step loss: 0.1235739067196846
7800 step loss: 0.12357384711503983


In [12]:
for n in np.arange(-1, 1.1, 0.2):
    n = round(n,1)
    value = LR.run([n])
    print(n, value)


-1.0 [[0.04909201]]
-0.8 [[0.04906187]]
-0.6 [[0.04903173]]
-0.4 [[0.04900159]]
-0.2 [[0.04897144]]
-0.0 [[0.0489413]]
0.2 [[0.04891116]]
0.4 [[0.04888101]]
0.6 [[0.04885087]]
0.8 [[0.04882073]]
1.0 [[0.04879059]]


In [10]:
value = LR.run([0])
print(value)

[[0.24643055]]


In [15]:
from pop import Pilot
import numpy as np
import time
import json


Car = Pilot.AutoCar()


# =========================
# 안전 설정값
# =========================

SPEED = 70          # 기존 70보다 낮게 시작 권장
MOVE_TIME = 0.35    # 앞으로 움직이는 시간
BACK_TIME = 0.35    # 뒤로 복귀하는 시간
STABLE_TIME = 0.15  # 정지 후 안정 시간

GYRO_COUNT = 8
GYRO_DELAY = 0.03


def car_forward(car, speed):
    try:
        car.forward(speed)
    except TypeError:
        car.forward()


def car_backward(car, speed):
    try:
        car.backward(speed)
    except TypeError:
        car.backward()


def get_gyro_avg(car, count=8, delay=0.03):
    values = []

    for _ in range(count):
        z = car.getGyro("z")
        values.append(float(z))
        time.sleep(delay)

    return sum(values) / len(values)


def move_forward_and_measure(car, steer, speed, move_time):
    """
    짧게 앞으로 이동하면서 gyro 평균을 측정
    """
    car.steering = steer
    car_forward(car, speed)

    start_time = time.time()
    gyro_values = []

    while time.time() - start_time < move_time:
        z = car.getGyro("z")
        gyro_values.append(float(z))
        time.sleep(GYRO_DELAY)

    car.stop()
    time.sleep(STABLE_TIME)

    if len(gyro_values) == 0:
        return 0.0

    return sum(gyro_values) / len(gyro_values)


def move_backward_return(car, steer, speed, back_time):
    """
    같은 steering 값으로 뒤로 복귀
    같은 steering을 유지해야 앞으로 간 곡선 경로를 되돌아오기 쉽다.
    """
    car.steering = steer
    car_backward(car, speed)

    time.sleep(back_time)

    car.stop()
    time.sleep(STABLE_TIME)


def collect_steering_data(car):
    dataset = {
        "steer": [],
        "gyro": []
    }

    car.setSpeed(SPEED)

    steer_values = np.linspace(-1.0, 1.0, 11)

    try:
        for steer in steer_values:
            steer = round(float(steer), 1)

            print("test steer:", steer)

            # 1. 앞으로 짧게 이동하면서 gyro 측정
            gyro_avg = move_forward_and_measure(
                car=car,
                steer=steer,
                speed=SPEED,
                move_time=MOVE_TIME
            )

            # 2. 같은 steering으로 뒤로 복귀
            move_backward_return(
                car=car,
                steer=steer,
                speed=SPEED,
                back_time=BACK_TIME
            )

            dataset["steer"].append(steer)
            dataset["gyro"].append(gyro_avg)

            print({
                "steer": steer,
                "gyro": gyro_avg
            })

            time.sleep(0.3)

    finally:
        car.stop()

    return dataset


class SteeringCalibrator:
    def __init__(self):
        self.steer_table = []
        self.gyro_table = []

        self.base_gyro = 0.0
        self.zero_correction = 0.0

        self.usable_gyro = 0.0

        self.left_steer = []
        self.left_delta = []

        self.right_steer = []
        self.right_delta = []

    def fit(self, steer_list, gyro_list):
        steer_arr = np.array(steer_list, dtype=float)
        gyro_arr = np.array(gyro_list, dtype=float)

        order = np.argsort(steer_arr)
        steer_arr = steer_arr[order]
        gyro_arr = gyro_arr[order]

        self.steer_table = steer_arr.tolist()
        self.gyro_table = gyro_arr.tolist()

        # steer=0에 가장 가까운 값의 gyro를 직진 기준으로 사용
        zero_idx = np.argmin(np.abs(steer_arr))
        self.base_gyro = float(gyro_arr[zero_idx])

        # 기준 gyro 대비 변화량
        delta_arr = gyro_arr - self.base_gyro

        # 실제로 가장 직진에 가까운 steering 값
        zero_correction_idx = np.argmin(np.abs(delta_arr))
        self.zero_correction = float(steer_arr[zero_correction_idx])

        left_mask = steer_arr < self.zero_correction
        right_mask = steer_arr > self.zero_correction

        left_steer = steer_arr[left_mask]
        left_delta = delta_arr[left_mask]

        right_steer = steer_arr[right_mask]
        right_delta = delta_arr[right_mask]

        if len(left_steer) < 2 or len(right_steer) < 2:
            raise ValueError("왼쪽/오른쪽 조향 데이터가 부족합니다.")

        self.left_steer = left_steer.tolist()
        self.left_delta = left_delta.tolist()

        self.right_steer = right_steer.tolist()
        self.right_delta = right_delta.tolist()

        left_max = max(abs(v) for v in self.left_delta)
        right_max = max(abs(v) for v in self.right_delta)

        # 좌우 모두 표현 가능한 공통 회전량
        self.usable_gyro = float(min(left_max, right_max))

        print("===== Calibration Result =====")
        print("base_gyro       :", self.base_gyro)
        print("zero_correction :", self.zero_correction)
        print("left_max_delta  :", left_max)
        print("right_max_delta :", right_max)
        print("usable_gyro     :", self.usable_gyro)

    def steering(self, desired_steer):
        """
        사용자가 원하는 steer -1.0 ~ 1.0을 넣으면
        실제 Car.steering에 넣을 보정값을 반환한다.
        """

        desired_steer = float(desired_steer)
        desired_steer = max(-1.0, min(1.0, desired_steer))

        if abs(desired_steer) < 1e-6:
            return self.zero_correction

        target_delta_abs = abs(desired_steer) * self.usable_gyro

        if desired_steer > 0:
            corrected = self._inverse_interpolate(
                target_delta_abs,
                self.right_steer,
                self.right_delta
            )
        else:
            corrected = self._inverse_interpolate(
                target_delta_abs,
                self.left_steer,
                self.left_delta
            )

        return max(-1.0, min(1.0, corrected))

    def _inverse_interpolate(self, target_delta_abs, steer_list, delta_list):
        steer_arr = np.array(steer_list, dtype=float)
        delta_abs_arr = np.abs(np.array(delta_list, dtype=float))

        order = np.argsort(delta_abs_arr)
        delta_abs_arr = delta_abs_arr[order]
        steer_arr = steer_arr[order]

        unique_delta = []
        unique_steer = []

        for d, s in zip(delta_abs_arr, steer_arr):
            if len(unique_delta) == 0 or abs(d - unique_delta[-1]) > 1e-6:
                unique_delta.append(d)
                unique_steer.append(s)

        if len(unique_delta) < 2:
            idx = np.argmin(np.abs(delta_abs_arr - target_delta_abs))
            return float(steer_arr[idx])

        return float(np.interp(
            target_delta_abs,
            unique_delta,
            unique_steer
        ))

    def save(self, path="steering_calibration.json"):
        data = {
            "steer_table": self.steer_table,
            "gyro_table": self.gyro_table,
            "base_gyro": self.base_gyro,
            "zero_correction": self.zero_correction,
            "usable_gyro": self.usable_gyro
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    def load(self, path="steering_calibration.json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        self.fit(data["steer_table"], data["gyro_table"])


def main():
    dataset = collect_steering_data(Car)

    print("===== Raw Dataset =====")
    print(dataset)

    calib = SteeringCalibrator()
    calib.fit(dataset["steer"], dataset["gyro"])
    calib.save()

    print("===== Calibrated Steering Table =====")

    for desired in np.linspace(-1.0, 1.0, 11):
        desired = round(float(desired), 1)
        corrected = calib.steering(desired)

        print({
            "desired": desired,
            "corrected": corrected
        })

    # 테스트 주행
    try:
        for desired in [-1.0, -0.5, 0.0, 0.5, 1.0]:
            corrected = calib.steering(desired)

            print("run:", {
                "desired": desired,
                "corrected": corrected
            })

            Car.steering = corrected
            car_forward(Car, SPEED)
            time.sleep(0.4)
            Car.stop()
            time.sleep(0.2)

            Car.steering = corrected
            car_backward(Car, SPEED)
            time.sleep(0.4)
            Car.stop()
            time.sleep(0.5)

    finally:
        Car.stop()


if __name__ == "__main__":
    main()

test steer: -1.0
{'steer': -1.0, 'gyro': 1593.0}
test steer: -0.8
{'steer': -0.8, 'gyro': 1498.5714285714287}
test steer: -0.6
{'steer': -0.6, 'gyro': 1230.857142857143}
test steer: -0.4
{'steer': -0.4, 'gyro': 987.1428571428571}
test steer: -0.2
{'steer': -0.2, 'gyro': 604.4285714285714}
test steer: 0.0
{'steer': 0.0, 'gyro': 269.85714285714283}
test steer: 0.2
{'steer': 0.2, 'gyro': -48.57142857142857}
test steer: 0.4
{'steer': 0.4, 'gyro': -383.42857142857144}
test steer: 0.6
{'steer': 0.6, 'gyro': -720.2857142857143}
test steer: 0.8
{'steer': 0.8, 'gyro': -948.8571428571429}
test steer: 1.0
{'steer': 1.0, 'gyro': -1342.5714285714287}
===== Raw Dataset =====
{'steer': [-1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0], 'gyro': [1593.0, 1498.5714285714287, 1230.857142857143, 987.1428571428571, 604.4285714285714, 269.85714285714283, -48.57142857142857, -383.42857142857144, -720.2857142857143, -948.8571428571429, -1342.5714285714287]}
===== Calibration Result =====
base_gyro 

In [21]:
from pop import Pilot, AI
import numpy as np
import time
import math


# ============================================================
# AutoCar 객체 생성
# ============================================================

Car = Pilot.AutoCar()


# ============================================================
# 기본 설정값
# ============================================================

SPEED = 70
MOVE_TIME = 0.7
BACK_TIME = 0.7
STABLE_TIME = 0.2
GYRO_DELAY = 0.03

CALIB_STEERS = np.linspace(-1.0, 1.0, 11)


# ============================================================
# gyro -> 조향각 추정 설정값
# ============================================================

# getGyro("z")가 raw 값이면 보통 1/131 사용
# getGyro("z")가 이미 deg/s라면 1.0으로 변경
GYRO_SCALE = 1.0 / 131.0

# 차량 앞뒤 바퀴 사이 거리, 실제 차량에 맞게 수정
WHEEL_BASE_CM = 18.0

# MOVE_TIME 동안 실제 이동한 거리
# 직접 자로 재서 넣으면 더 정확하다.
# 예: speed=70, move_time=0.7 동안 35cm 이동하면 35.0
MOVE_DISTANCE_CM = 35.0

# 조향각 표시용 최대 바퀴 각도
# AutoCar.steering = 1.0 일 때 앞바퀴가 대략 몇 도 꺾이는지
MAX_WHEEL_ANGLE_DEG = 30.0


# ============================================================
# 조향 범위 보정 설정
# ============================================================

RANGE_GAIN = 1.6

LEFT_GAIN = RANGE_GAIN
RIGHT_GAIN = RANGE_GAIN

STEER_MIN = -1.0
STEER_MAX = 1.0


# ============================================================
# AutoCar 호환 함수
# ============================================================

def car_forward(car, speed):
    try:
        car.forward(speed)
    except TypeError:
        car.forward()


def car_backward(car, speed):
    try:
        car.backward(speed)
    except TypeError:
        car.backward()


# ============================================================
# Linear Regression 관련 함수
# ============================================================

def train_lr(lr, times=1000):
    if hasattr(lr, "train"):
        try:
            lr.train(times=times)
        except TypeError:
            lr.train()
    elif hasattr(lr, "fit"):
        try:
            lr.fit(times=times)
        except TypeError:
            lr.fit()
    else:
        raise RuntimeError("Linear_Regression 객체에 train 또는 fit 메서드가 없습니다.")


def lr_value(lr, x):
    result = lr.run([float(x)])

    try:
        return float(result[0][0])
    except Exception:
        try:
            return float(result[0])
        except Exception:
            return float(result)


# ============================================================
# gyro 값으로 차체 회전각 / 바퀴 조향각 계산
# ============================================================

def calc_angle_from_gyro_samples(gyro_samples, dt_samples):
    """
    gyro_samples:
        getGyro("z")에서 읽은 값들

    dt_samples:
        각 gyro 샘플 사이의 시간 간격들

    반환:
        gyro_avg_raw
        gyro_avg_dps
        yaw_angle_deg
        wheel_angle_deg
    """

    if len(gyro_samples) == 0:
        return {
            "gyro_avg_raw": 0.0,
            "gyro_avg_dps": 0.0,
            "yaw_angle_deg": 0.0,
            "wheel_angle_deg": 0.0
        }

    gyro_raw_arr = np.array(gyro_samples, dtype=float)
    dt_arr = np.array(dt_samples, dtype=float)

    gyro_avg_raw = float(np.mean(gyro_raw_arr))

    # raw gyro 값을 deg/s로 변환
    gyro_dps_arr = gyro_raw_arr * GYRO_SCALE
    gyro_avg_dps = float(np.mean(gyro_dps_arr))

    # yaw angle = gyro_z 각속도를 시간 적분
    if len(dt_arr) == len(gyro_dps_arr):
        yaw_angle_deg = float(np.sum(gyro_dps_arr * dt_arr))
    else:
        yaw_angle_deg = float(gyro_avg_dps * MOVE_TIME)

    # 차체 회전각 yaw로부터 조향각 추정
    # bicycle model 근사:
    # radius = distance / yaw_rad
    # steering_angle = atan(wheel_base / radius)
    yaw_rad = math.radians(yaw_angle_deg)

    if abs(MOVE_DISTANCE_CM) < 1e-6:
        wheel_angle_deg = 0.0
    else:
        wheel_angle_rad = math.atan2(WHEEL_BASE_CM * yaw_rad, MOVE_DISTANCE_CM)
        wheel_angle_deg = math.degrees(wheel_angle_rad)

    return {
        "gyro_avg_raw": gyro_avg_raw,
        "gyro_avg_dps": gyro_avg_dps,
        "yaw_angle_deg": yaw_angle_deg,
        "wheel_angle_deg": wheel_angle_deg
    }


def estimate_wheel_angle_from_steer(steer):
    """
    steering 명령값 기준의 단순 바퀴 각도 추정.
    실제 측정값이 아니라 명령값 기반 표시용이다.
    """
    return float(steer) * MAX_WHEEL_ANGLE_DEG


# ============================================================
# 앞으로 움직이면서 gyro, yaw angle, wheel angle 측정
# ============================================================

def move_forward_and_measure(car, steer, speed, move_time):
    """
    지정한 steer 값으로 앞으로 움직이면서
    gyro 평균, 차체 회전각, 추정 바퀴 조향각을 함께 계산한다.
    """

    car.steering = steer
    car_forward(car, speed)

    start = time.time()
    last = start

    gyro_samples = []
    dt_samples = []

    while time.time() - start < move_time:
        now = time.time()
        dt = now - last
        last = now

        z = car.getGyro("z")

        gyro_samples.append(float(z))
        dt_samples.append(float(dt))

        time.sleep(GYRO_DELAY)

    car.stop()
    time.sleep(STABLE_TIME)

    angle_info = calc_angle_from_gyro_samples(
        gyro_samples=gyro_samples,
        dt_samples=dt_samples
    )

    result = {
        "steer": float(steer),
        "command_wheel_angle_deg": estimate_wheel_angle_from_steer(steer),
        "gyro": angle_info["gyro_avg_raw"],
        "gyro_dps": angle_info["gyro_avg_dps"],
        "yaw_angle_deg": angle_info["yaw_angle_deg"],
        "wheel_angle_deg": angle_info["wheel_angle_deg"]
    }

    return result


def move_backward_return(car, steer, speed, back_time):
    car.steering = steer
    car_backward(car, speed)

    time.sleep(back_time)

    car.stop()
    time.sleep(STABLE_TIME)


# ============================================================
# 1. steer -> gyro / angle 반응 데이터 수집
# ============================================================

def collect_response_data(car):
    dataset = {
        "steer": [],
        "gyro": [],
        "gyro_dps": [],
        "yaw_angle_deg": [],
        "wheel_angle_deg": [],
        "command_wheel_angle_deg": []
    }

    car.setSpeed(SPEED)

    try:
        for steer in CALIB_STEERS:
            steer = round(float(steer), 1)

            print("측정 steer:", steer)

            measured = move_forward_and_measure(
                car=car,
                steer=steer,
                speed=SPEED,
                move_time=MOVE_TIME
            )

            move_backward_return(
                car=car,
                steer=steer,
                speed=SPEED,
                back_time=BACK_TIME
            )

            dataset["steer"].append(measured["steer"])
            dataset["gyro"].append(measured["gyro"])
            dataset["gyro_dps"].append(measured["gyro_dps"])
            dataset["yaw_angle_deg"].append(measured["yaw_angle_deg"])
            dataset["wheel_angle_deg"].append(measured["wheel_angle_deg"])
            dataset["command_wheel_angle_deg"].append(measured["command_wheel_angle_deg"])

            print({
                "steer": measured["steer"],
                "command_wheel_angle_deg": measured["command_wheel_angle_deg"],
                "gyro_raw": measured["gyro"],
                "gyro_dps": measured["gyro_dps"],
                "yaw_angle_deg": measured["yaw_angle_deg"],
                "estimated_wheel_angle_deg": measured["wheel_angle_deg"]
            })

            time.sleep(0.3)

    finally:
        car.stop()

    return dataset


# ============================================================
# gyro=0이 되는 실제 직진 조향값 찾기
# ============================================================

def find_zero_steer(steer_arr, gyro_arr):
    for i in range(len(steer_arr) - 1):
        s1 = steer_arr[i]
        s2 = steer_arr[i + 1]

        g1 = gyro_arr[i]
        g2 = gyro_arr[i + 1]

        if g1 == 0:
            return float(s1)

        if g1 * g2 < 0:
            zero_steer = s1 + (0 - g1) * (s2 - s1) / (g2 - g1)
            return float(zero_steer)

    idx = np.argmin(np.abs(gyro_arr))
    return float(steer_arr[idx])


# ============================================================
# 2. 측정 데이터로 AI 학습 데이터 만들기
# ============================================================

def make_training_data(dataset):
    steer_arr = np.array(dataset["steer"], dtype=float)
    gyro_arr = np.array(dataset["gyro"], dtype=float)

    order = np.argsort(steer_arr)
    steer_arr = steer_arr[order]
    gyro_arr = gyro_arr[order]

    print("===== 정렬된 측정 데이터 =====")

    for i in range(len(steer_arr)):
        print({
            "steer": float(steer_arr[i]),
            "gyro": float(gyro_arr[i])
        })

    zero_steer = find_zero_steer(steer_arr, gyro_arr)

    print("실제 직진 보정 steer:", zero_steer)
    print("실제 직진 명령 기준 바퀴각 추정:", estimate_wheel_angle_from_steer(zero_steer))

    left_mask = gyro_arr > 0
    right_mask = gyro_arr < 0

    left_steer = steer_arr[left_mask]
    left_gyro_abs = np.abs(gyro_arr[left_mask])

    right_steer = steer_arr[right_mask]
    right_gyro_abs = np.abs(gyro_arr[right_mask])

    left_steer = np.append(left_steer, zero_steer)
    left_gyro_abs = np.append(left_gyro_abs, 0.0)

    right_steer = np.append(right_steer, zero_steer)
    right_gyro_abs = np.append(right_gyro_abs, 0.0)

    left_order = np.argsort(left_gyro_abs)
    right_order = np.argsort(right_gyro_abs)

    left_gyro_abs = left_gyro_abs[left_order]
    left_steer = left_steer[left_order]

    right_gyro_abs = right_gyro_abs[right_order]
    right_steer = right_steer[right_order]

    left_max = np.max(left_gyro_abs)
    right_max = np.max(right_gyro_abs)

    usable_gyro = min(left_max, right_max)

    print("left_max_gyro :", left_max)
    print("right_max_gyro:", right_max)
    print("usable_gyro   :", usable_gyro)

    desired_values = np.linspace(-1.0, 1.0, 11)

    X_data = []
    Y_data = []

    print("===== 보정 테이블 =====")

    for desired in desired_values:
        desired = round(float(desired), 1)

        if desired < 0:
            target_gyro = abs(desired) * usable_gyro

            corrected = np.interp(
                target_gyro,
                left_gyro_abs,
                left_steer
            )

        elif desired > 0:
            target_gyro = abs(desired) * usable_gyro

            corrected = np.interp(
                target_gyro,
                right_gyro_abs,
                right_steer
            )

        else:
            corrected = zero_steer

        corrected = float(corrected)
        corrected = max(STEER_MIN, min(STEER_MAX, corrected))

        X_data.append([desired])
        Y_data.append([corrected])

        print({
            "desired_steer": desired,
            "corrected_steer": corrected,
            "corrected_command_wheel_angle_deg": estimate_wheel_angle_from_steer(corrected)
        })

    return zero_steer, X_data, Y_data


# ============================================================
# 3. AI Linear Regression 학습
# ============================================================

def train_alignment_model(X_data, Y_data):
    LR = AI.Linear_Regression()

    try:
        LR.restore = False
    except Exception:
        pass

    try:
        LR.ckpt_name = "wheel_alignment_" + str(int(time.time()))
    except Exception:
        pass

    LR.X_data = X_data
    LR.Y_data = Y_data

    train_lr(LR, times=1000)

    return LR


# ============================================================
# 4. 보정 조향값 계산
# ============================================================

def calibrated_steering(LR, desired_steer, zero_steer):
    desired_steer = float(desired_steer)
    desired_steer = max(-1.0, min(1.0, desired_steer))

    lr_corrected = lr_value(LR, desired_steer)

    if desired_steer < 0:
        gain = LEFT_GAIN
    elif desired_steer > 0:
        gain = RIGHT_GAIN
    else:
        gain = 1.0

    corrected = zero_steer + gain * (lr_corrected - zero_steer)
    corrected = max(STEER_MIN, min(STEER_MAX, corrected))

    return corrected


# ============================================================
# 5. AI 보정 결과 확인
# ============================================================

def print_ai_result(LR, zero_steer):
    print("===== AI 보정 결과 확인 =====")

    for desired in np.linspace(-1.0, 1.0, 11):
        desired = round(float(desired), 1)

        lr_corrected = lr_value(LR, desired)

        final_corrected = calibrated_steering(
            LR=LR,
            desired_steer=desired,
            zero_steer=zero_steer
        )

        print({
            "desired": desired,
            "LR_corrected": lr_corrected,
            "final_corrected": final_corrected,
            "command_wheel_angle_deg": estimate_wheel_angle_from_steer(final_corrected)
        })


# ============================================================
# 6. 보정된 조향값으로 gyro / 조향각 검증
# ============================================================

def verify_alignment_with_gyro(car, LR, zero_steer):
    print("===== Gyro 기반 얼라인먼트 검증 =====")

    test_values = [-1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

    result = {
        "desired": [],
        "corrected": [],
        "gyro": [],
        "gyro_dps": [],
        "yaw_angle_deg": [],
        "wheel_angle_deg": [],
        "command_wheel_angle_deg": []
    }

    try:
        for desired in test_values:
            desired = round(float(desired), 1)

            corrected = calibrated_steering(
                LR=LR,
                desired_steer=desired,
                zero_steer=zero_steer
            )

            measured = move_forward_and_measure(
                car=car,
                steer=corrected,
                speed=SPEED,
                move_time=MOVE_TIME
            )

            move_backward_return(
                car=car,
                steer=corrected,
                speed=SPEED,
                back_time=BACK_TIME
            )

            result["desired"].append(desired)
            result["corrected"].append(corrected)
            result["gyro"].append(measured["gyro"])
            result["gyro_dps"].append(measured["gyro_dps"])
            result["yaw_angle_deg"].append(measured["yaw_angle_deg"])
            result["wheel_angle_deg"].append(measured["wheel_angle_deg"])
            result["command_wheel_angle_deg"].append(measured["command_wheel_angle_deg"])

            print({
                "desired": desired,
                "corrected": corrected,
                "command_wheel_angle_deg": measured["command_wheel_angle_deg"],
                "gyro_raw": measured["gyro"],
                "gyro_dps": measured["gyro_dps"],
                "yaw_angle_deg": measured["yaw_angle_deg"],
                "estimated_wheel_angle_deg": measured["wheel_angle_deg"]
            })

            time.sleep(0.3)

    finally:
        car.stop()

    print("===== 좌우 대칭성 확인 =====")

    for level in [0.2, 0.4, 0.6, 0.8, 1.0]:
        left_desired = round(-level, 1)
        right_desired = round(level, 1)

        if left_desired not in result["desired"]:
            continue

        if right_desired not in result["desired"]:
            continue

        left_idx = result["desired"].index(left_desired)
        right_idx = result["desired"].index(right_desired)

        left_gyro = result["gyro"][left_idx]
        right_gyro = result["gyro"][right_idx]

        left_yaw = result["yaw_angle_deg"][left_idx]
        right_yaw = result["yaw_angle_deg"][right_idx]

        left_wheel_angle = result["wheel_angle_deg"][left_idx]
        right_wheel_angle = result["wheel_angle_deg"][right_idx]

        left_abs = abs(left_gyro)
        right_abs = abs(right_gyro)

        diff = abs(left_abs - right_abs)
        avg = (left_abs + right_abs) / 2

        if avg == 0:
            error_rate = 0.0
        else:
            error_rate = diff / avg * 100

        print({
            "level": level,
            "left_gyro": left_gyro,
            "right_gyro": right_gyro,
            "left_yaw_angle_deg": left_yaw,
            "right_yaw_angle_deg": right_yaw,
            "left_estimated_wheel_angle_deg": left_wheel_angle,
            "right_estimated_wheel_angle_deg": right_wheel_angle,
            "gyro_error_rate_percent": error_rate
        })

    return result


# ============================================================
# 7. 실제 주행 테스트
# ============================================================

def run_test(car, LR, zero_steer):
    print("===== 실제 주행 테스트 =====")

    try:
        for desired in [-1.0, -0.5, 0.0, 0.5, 1.0]:
            corrected = calibrated_steering(
                LR=LR,
                desired_steer=desired,
                zero_steer=zero_steer
            )

            print({
                "desired": desired,
                "corrected": corrected,
                "command_wheel_angle_deg": estimate_wheel_angle_from_steer(corrected)
            })

            car.steering = corrected
            car_forward(car, SPEED)
            time.sleep(MOVE_TIME)

            car.stop()
            time.sleep(0.2)

            car.steering = corrected
            car_backward(car, SPEED)
            time.sleep(BACK_TIME)

            car.stop()
            time.sleep(0.5)

    finally:
        car.stop()


# ============================================================
# 전체 실행
# ============================================================

def main():
    dataset = collect_response_data(Car)

    print("===== 측정 데이터 =====")
    print(dataset)

    zero_steer, X_data, Y_data = make_training_data(dataset)

    print("===== AI 학습 데이터 =====")
    print("X_data:", X_data)
    print("Y_data:", Y_data)

    LR = train_alignment_model(X_data, Y_data)

    print_ai_result(LR, zero_steer)

    verify_result = verify_alignment_with_gyro(
        car=Car,
        LR=LR,
        zero_steer=zero_steer
    )

    run_test(Car, LR, zero_steer)


if __name__ == "__main__":
    main()

측정 steer: -1.0
{'steer': -1.0, 'command_wheel_angle_deg': -30.0, 'gyro_raw': 1695.7692307692307, 'gyro_dps': 12.944803288314738, 'yaw_angle_deg': 8.919497016732018, 'estimated_wheel_angle_deg': 4.577406470751218}
측정 steer: -0.8
{'steer': -0.8, 'command_wheel_angle_deg': -24.0, 'gyro_raw': 2022.0, 'gyro_dps': 15.435114503816793, 'yaw_angle_deg': 10.498702591612137, 'estimated_wheel_angle_deg': 5.383434552735027}
측정 steer: -0.6
{'steer': -0.6, 'command_wheel_angle_deg': -18.0, 'gyro_raw': 1693.923076923077, 'gyro_dps': 12.93071051086318, 'yaw_angle_deg': 8.844788014433767, 'estimated_wheel_angle_deg': 4.5392273743888465}
측정 steer: -0.4
{'steer': -0.4, 'command_wheel_angle_deg': -12.0, 'gyro_raw': 1270.1538461538462, 'gyro_dps': 9.695830886670581, 'yaw_angle_deg': 6.595234909130418, 'estimated_wheel_angle_deg': 3.387881194994177}
측정 steer: -0.2
{'steer': -0.2, 'command_wheel_angle_deg': -6.0, 'gyro_raw': 792.3846153846154, 'gyro_dps': 6.048737522019963, 'yaw_angle_deg': 4.111203546742447,